# Psalm Sentiment Classification: Model Comparison

Extends `02_sentiment_nb.ipynb` by testing five classifiers across two feature families (bag-of-words and sentence embeddings) to maximise accuracy on the verse-level sentiment task.

In [1]:
import numpy as np
import pandas as pd

import psalm_scraper as ps
import psalm_utils as pu

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score

from sentence_transformers import SentenceTransformer

c:\Users\hmhol\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ps_dict = ps.load_ps_dict('ps_verses.json')

training_data = [
    (v['text'], v['label'])
    for p in ps_dict
    for v in ps_dict[p]
    if v['label'] is not None
]
texts, labels = zip(*training_data)
texts  = np.array(texts)
labels = np.array(labels)

print(f'Total samples:  {len(texts)}')
print(f'Positive (1):   {int(labels.sum())} ({labels.mean():.1%})')
print(f'Negative (0):   {int((1-labels).sum())} ({(1-labels).mean():.1%})')

Total samples:  1172
Positive (1):   817 (69.7%)
Negative (0):   355 (30.3%)


In [3]:
# Stratified split ensures both sets have the same positive/negative ratio
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

results = {}

def evaluate(name, y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob if y_prob is not None else y_pred)
    results[name] = {'Accuracy': acc, 'F1': f1, 'Recall': rec, 'ROC AUC': auc}
    print(f'\n{name}')
    print(f'  Accuracy: {acc:.4f}')
    print(f'  F1:       {f1:.4f}')
    print(f'  Recall:   {rec:.4f}')
    print(f'  ROC AUC:  {auc:.4f}')

Train: 937, Test: 235


## 1. Baseline: Naive Bayes

Direct replication of `02_sentiment_nb.ipynb` — bag-of-words with Laplace smoothing.

In [4]:
freqs = pu.build_freqs(X_train, y_train)
d_pos = y_train.sum()
d_neg = (1 - y_train).sum()
log_prior = np.log(d_pos / d_neg)

vocab = [x[0] for x in set(freqs.keys())]
V = len(set(vocab))

N_pos, N_neg = 0, 0
for pair in freqs:
    if pair[1] > 0:
        N_pos += freqs[pair]
    else:
        N_neg += freqs[pair]

loglikelihood = {}
for word in vocab:
    freq_pos = freqs.get((word, 1.0), 0)
    freq_neg = freqs.get((word, 0.0), 0)
    p_w_pos = (freq_pos + 1) / (N_pos + V)
    p_w_neg = (freq_neg + 1) / (N_neg + V)
    loglikelihood[word] = np.log(p_w_pos / p_w_neg)

nb_scores = np.array([pu.naive_bayes_predict(t, log_prior, loglikelihood) for t in X_test])
nb_pred   = (nb_scores > 0).astype(int)
nb_prob   = 1 / (1 + np.exp(-nb_scores))  # sigmoid to get probabilities

evaluate('Naive Bayes', y_test, nb_pred, nb_prob)


Naive Bayes
  Accuracy: 0.9149
  F1:       0.9412
  Recall:   0.9756
  ROC AUC:  0.9667


## 2. TF-IDF classifiers

TF-IDF weights rare-but-important words more heavily than raw counts. Using bigrams (`ngram_range=(1,2)`) to capture short phrases like "not wicked" or "great mercy".

In [5]:
pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True)),
    ('clf',  LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)
pred = pipe_lr.predict(X_test)
prob = pipe_lr.predict_proba(X_test)[:, 1]
evaluate('TF-IDF + Logistic Regression', y_test, pred, prob)


TF-IDF + Logistic Regression
  Accuracy: 0.8043
  F1:       0.8763
  Recall:   0.9939
  ROC AUC:  0.9490


In [6]:
pipe_svm = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True)),
    ('clf',  CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000, random_state=42)))
])
pipe_svm.fit(X_train, y_train)
pred = pipe_svm.predict(X_test)
prob = pipe_svm.predict_proba(X_test)[:, 1]
evaluate('TF-IDF + Linear SVM', y_test, pred, prob)


TF-IDF + Linear SVM
  Accuracy: 0.8979
  F1:       0.9298
  Recall:   0.9695
  ROC AUC:  0.9699


## 3. Sentence Transformer embeddings

Encode each verse into a 384-dimensional semantic vector using `all-MiniLM-L6-v2` (the same model used in `03_theme_clustering.ipynb`). L2-normalise before fitting.

In [7]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')
X_train_emb = encoder.encode(X_train.tolist(), show_progress_bar=True)
X_test_emb  = encoder.encode(X_test.tolist(),  show_progress_bar=True)
X_train_emb = normalize(X_train_emb)
X_test_emb  = normalize(X_test_emb)
print(f'Embedding shape: {X_train_emb.shape}')

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Batches:   3%|▎         | 1/30 [00:00<00:03,  7.74it/s]

Batches:  10%|█         | 3/30 [00:00<00:02, 11.29it/s]

Batches:  17%|█▋        | 5/30 [00:00<00:02, 12.30it/s]

Batches:  23%|██▎       | 7/30 [00:00<00:01, 13.62it/s]

...

Batches:  25%|██▌       | 2/8 [00:00<00:00, 12.93it/s]

Batches:  50%|█████     | 4/8 [00:00<00:00, 14.68it/s]

Batches:  88%|████████▊ | 7/8 [00:00<00:00, 17.81it/s]

Batches: 100%|██████████| 8/8 [00:00<00:00, 18.15it/s]

Embedding shape: (937, 384)


In [8]:
lr_emb = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_emb.fit(X_train_emb, y_train)
pred = lr_emb.predict(X_test_emb)
prob = lr_emb.predict_proba(X_test_emb)[:, 1]
evaluate('Sentence Transformer + Logistic Regression', y_test, pred, prob)


Sentence Transformer + Logistic Regression
  Accuracy: 0.9234
  F1:       0.9458
  Recall:   0.9573
  ROC AUC:  0.9757


In [9]:
svm_emb = CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000, random_state=42))
svm_emb.fit(X_train_emb, y_train)
pred = svm_emb.predict(X_test_emb)
prob = svm_emb.predict_proba(X_test_emb)[:, 1]
evaluate('Sentence Transformer + Linear SVM', y_test, pred, prob)


Sentence Transformer + Linear SVM
  Accuracy: 0.9234
  F1:       0.9451
  Recall:   0.9451
  ROC AUC:  0.9735


## 4. Results summary

In [10]:
results_df = pd.DataFrame(results).T.sort_values('Accuracy', ascending=False)
print(results_df.to_string(float_format=lambda x: f'{x:.4f}'))

                                            Accuracy     F1  Recall  ROC AUC
Sentence Transformer + Linear SVM             0.9234 0.9451  0.9451   0.9735
Sentence Transformer + Logistic Regression    0.9234 0.9458  0.9573   0.9757
Naive Bayes                                   0.9149 0.9412  0.9756   0.9667
TF-IDF + Linear SVM                           0.8979 0.9298  0.9695   0.9699
TF-IDF + Logistic Regression                  0.8043 0.8763  0.9939   0.9490
